# End-to-end two-group retinal analysis

## Quick start

1. Attach this notebook to GPU compute for the full run. CPU/serverless is
   sufficient when `stop_after_stage=quality`.
2. In the widgets, provide an image manifest, image directory, or ZIP
   archive plus a cohort table containing participant ID, group, and age.
3. Set `group_a_value` and `group_b_value`; group B is the modeled positive
   class and is matched to group A.
4. Give the analysis a stable `analysis_name`. Leave `run_id` blank and
   keep `resume_latest=true`; reruns then resume the newest compatible run.
5. Run all cells. Quality, embeddings, segmentation, and explainability
   save validated Parquet batches as they finish. If the kernel stops,
   rerun the notebook and completed batches will be skipped.
6. Review `RUN_README.md`, aggregate tables, and figures in the printed
   run directory. Files containing participant IDs are labeled `private`.

Existing quality manifests, RETFound embeddings, or segmentations may be
supplied to bypass completed work. Passwords and Hugging Face tokens are
temporary widgets and are never written to the run configuration.

## Accepted inputs and analysis contract

**Images:** JPEG/JPG, PNG, TIFF, or DICOM files. They can be supplied as a
directory, a manifest, or one or more ZIP files separated by semicolons.
Password-protected ZIP files are supported. DICOM pixels are decoded and
materialized as RGB JPEGs before quality assessment; JPEG-lossless DICOM
requires the installed `pylibjpeg` plugins.

**Image manifests:** Delta, Parquet, or CSV. `image_path` is required unless
`image_path_column` names a different field. Participant ID and eye can be
supplied as columns or parsed from paths with the configured regular
expressions. Optional visit, sex, and device fields are retained.

**Cohort tables:** Delta, Parquet, or CSV. Required fields are participant
ID, a two-level group variable, and age at imaging. If `cohort_path` is
empty, those fields must already be present in the image manifest.

**Reusable artifacts:** Quality, embedding, and segmentation results may be
Delta, Parquet, or a directory of Parquet batches. Existing embeddings must
contain one 1,024-element vector per image and must have been produced by a
compatible RETFound checkpoint.

Matching and evaluation are participant-level. Eyes from one participant
never cross model folds, and an entire matched set remains within one fold.
The primary classifier is a regularized linear head on frozen RETFound
vectors so its patch-level attribution is exactly decomposable. If device
and group are perfectly confounded, the report explicitly states that a
disease-specific conclusion is not identifiable from this analysis.

In [ ]:
%pip install -q \
  "numpy>=2.0,<2.3" "pandas>=2.2,<3" "pyarrow>=15" \
  "scikit-learn>=1.5,<2" "scipy>=1.11,<2" \
  "matplotlib>=3.8" "seaborn>=0.13" "joblib>=1.3" \
  "timm>=1.0,<2" "huggingface_hub>=0.24" "pyzipper>=0.3.6" \
  "pydicom>=3.0" "pylibjpeg>=2.0" "pylibjpeg-libjpeg>=2.1" \
  "pylibjpeg-openjpeg>=2.4" \
  "git+https://github.com/berenslab/fundus_image_toolbox.git@d7757e28fbf639856b53cfe00019f605af8c1f17"

In [ ]:
dbutils.library.restartPython()

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import sys
import time
import uuid
import zipfile

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import scipy
import torch


def ensure_text_widget(name, default, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(name, default, choices, label):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


# Paths and run identity
ensure_text_widget(
    "repo_root",
    "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina",
    "Repository root",
)
ensure_text_widget(
    "output_base",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "two_group_retinal_analysis",
    "Output base",
)
ensure_text_widget("analysis_name", "group_a_vs_group_b", "Analysis name")
ensure_text_widget("run_id", "", "Existing run ID (optional)")
ensure_dropdown_widget("resume_latest", "true", ["true", "false"], "Resume latest")

# Input sources
ensure_dropdown_widget(
    "image_source_type", "manifest", ["manifest", "directory", "zip"],
    "Image source type",
)
ensure_text_widget("image_source_path", "", "Image source path(s)")
ensure_dropdown_widget(
    "image_manifest_format", "delta", ["delta", "parquet", "csv"],
    "Manifest format",
)
ensure_text_widget("cohort_path", "", "Cohort table path (optional)")
ensure_dropdown_widget(
    "cohort_format", "delta", ["delta", "parquet", "csv"],
    "Cohort format",
)
ensure_text_widget("existing_quality_path", "", "Existing quality path")
ensure_text_widget("existing_embeddings_path", "", "Existing embeddings path")
ensure_text_widget("existing_segmentations_path", "", "Existing segmentations path")
ensure_dropdown_widget(
    "existing_artifact_format", "parquet", ["delta", "parquet", "csv"],
    "Existing artifact format",
)

# Column names and parsing
ensure_text_widget("image_path_column", "image_path", "Image-path column")
ensure_text_widget("participant_id_column", "participant_id", "Participant-ID column")
ensure_text_widget("group_column", "group", "Group column")
ensure_text_widget("age_column", "age", "Age column")
ensure_text_widget("eye_column", "eye", "Eye column")
ensure_text_widget("visit_column", "visit", "Visit column")
ensure_text_widget("sex_column", "sex", "Sex column")
ensure_text_widget("device_column", "device", "Device/source column")
ensure_text_widget(
    "participant_id_regex", r"(?i)(?:^|/)(?P<participant_id>\d{7,10})(?:/|_)",
    "Participant-ID path regex",
)
ensure_text_widget(
    "eye_regex", r"(?i)(?P<eye>left|right|os|od)(?=\.[^.]+$)",
    "Eye path regex",
)

# Group and matching definitions
ensure_text_widget("group_a_value", "0", "Group A value")
ensure_text_widget("group_a_name", "Group A", "Group A name")
ensure_text_widget("group_b_value", "1", "Group B value")
ensure_text_widget("group_b_name", "Group B", "Group B name")
ensure_text_widget("age_caliper_years", "2.0", "Age caliper (years)")
ensure_text_widget("match_ratio", "1", "Group A per Group B")
ensure_text_widget(
    "exact_match_columns", "sex,visit", "Exact-match normalized columns",
)

# Batching and stage control
ensure_text_widget("manifest_batch_size", "500", "Manifest/quality batch size")
ensure_text_widget("retfound_tensor_batch_size", "8", "RETFound tensor batch")
ensure_text_widget("segmentation_batch_size", "4", "Segmentation batch size")
ensure_text_widget("explainability_batch_size", "50", "Explainability batch size")
ensure_text_widget(
    "max_new_batches_per_stage", "0", "New batches per stage (0=all)",
)
ensure_dropdown_widget(
    "stop_after_stage", "complete",
    ["quality", "matching", "embeddings", "segmentation", "model", "explainability", "complete"],
    "Stop after stage",
)
ensure_text_widget("force_stages", "", "Comma-separated stages to recompute")
ensure_dropdown_widget("save_preprocessed", "false", ["true", "false"], "Save crops")
ensure_dropdown_widget("run_segmentation", "true", ["true", "false"], "Run segmentation")
ensure_dropdown_widget("run_explainability", "true", ["true", "false"], "Run explainability")
ensure_dropdown_widget(
    "run_targeted_occlusion", "false", ["true", "false"], "Run targeted occlusion",
)
ensure_text_widget("maximum_explainability_images", "0", "Maximum XAI images (0=all)")
ensure_text_widget("occlusion_control_masks", "10", "Occlusion control masks")

# RETFound and temporary credentials
ensure_text_widget("retfound_repo", "", "Local RETFound repository (optional)")
ensure_text_widget("checkpoint_path", "", "RETFound checkpoint (optional)")
ensure_dropdown_widget("allow_repo_clone", "true", ["true", "false"], "Allow RETFound clone")
ensure_dropdown_widget("allow_checkpoint_download", "true", ["true", "false"], "Allow checkpoint download")
ensure_dropdown_widget("device", "auto", ["auto", "cuda", "cpu"], "Model device")
ensure_text_widget("hf_token", "", "Temporary Hugging Face token")
ensure_text_widget("archive_password", "", "Temporary ZIP password")

print("Widgets are ready. Edit values above the notebook, then rerun this cell onward.")

In [ ]:
def widget(name):
    return dbutils.widgets.get(name).strip()


def widget_bool(name):
    return widget(name).lower() == "true"


def normalized_fuse_path(value):
    value = str(value).strip()
    if value.startswith("dbfs:/Volumes/"):
        return "/Volumes/" + value[len("dbfs:/Volumes/"):]
    if value.startswith("dbfs:/"):
        return "/dbfs/" + value[len("dbfs:/"):]
    return value


repo_root = Path(normalized_fuse_path(widget("repo_root")))
module_root = repo_root / "src"
if not module_root.exists():
    raise FileNotFoundError(f"Repository source directory not found: {module_root}")
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

import importlib
import two_group_retinal_pipeline as _two_group_module
_two_group_module = importlib.reload(_two_group_module)

from age_gap_extremes import fundus_physiology_proxies
from clsa_anatomic_explainability import (
    attribution_region_metrics,
    build_anatomic_masks,
    disc_fovea_affine_matrix,
)
from fundus_retfound_pipeline import (
    QualityConfig,
    RETFoundConfig,
    exact_linear_patch_contributions,
    extract_retfound_embeddings,
    linear_head_score_from_array,
    load_retfound_model,
    prepare_model_input,
    preprocess_fundus,
    write_frame,
    write_json,
)
from glaucoma_classifier_spatial import sample_equal_area_control_masks
from two_group_retinal_pipeline import (
    MatchConfig,
    aggregate_participant_embeddings,
    batch_ranges,
    consolidate_batch_parquets,
    fit_grouped_oof_classifier,
    matched_set_permutation_inference,
    matching_balance,
    match_participants,
    prediction_metrics,
    slugify,
    stable_image_key,
)

analysis_name = widget("analysis_name")
analysis_slug = slugify(analysis_name)
output_base = Path(normalized_fuse_path(widget("output_base")))
output_base.mkdir(parents=True, exist_ok=True)
source_path_text = widget("image_source_path")
if not source_path_text:
    raise ValueError("image_source_path is required")

identity = {
    "analysis_name": analysis_name,
    "image_source_type": widget("image_source_type"),
    "image_source_path": source_path_text,
    "cohort_path": widget("cohort_path"),
    "group_column": widget("group_column"),
    "group_a_value": widget("group_a_value"),
    "group_b_value": widget("group_b_value"),
}
identity_sha256 = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode("utf-8")
).hexdigest()
latest_pointer = output_base / f"_LATEST_{analysis_slug}.json"
requested_run_id = widget("run_id")
resolved_run_id = requested_run_id or None
if not resolved_run_id and widget_bool("resume_latest") and latest_pointer.exists():
    candidate = json.loads(latest_pointer.read_text())
    if candidate.get("identity_sha256") == identity_sha256:
        resolved_run_id = candidate.get("run_id")
if not resolved_run_id:
    resolved_run_id = time.strftime(f"{analysis_slug}__%Y%m%d_%H%M%S")
run_root = output_base / slugify(resolved_run_id, default=analysis_slug)
run_root.mkdir(parents=True, exist_ok=True)

local_staging = Path(f"/local_disk0/tmp/two_group_retinal_{os.getpid()}")
local_staging.mkdir(parents=True, exist_ok=True)


def digest_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    size = 0
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            size += len(chunk)
            digest.update(chunk)
    return size, digest.hexdigest()


def publish_local_artifact(local_path, destination, retries=4):
    local_path = Path(local_path)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = digest_file(local_path)
    if destination.is_file():
        try:
            if digest_file(destination) == expected:
                return destination
        except OSError:
            pass
    last_error = None
    for attempt in range(1, retries + 1):
        partial = destination.with_name(
            f".{destination.name}.{uuid.uuid4().hex}.partial"
        )
        try:
            with local_path.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024 * 1024)
                target.flush()
                os.fsync(target.fileno())
            if digest_file(partial) != expected:
                raise OSError("Temporary Volume copy failed validation")
            os.replace(partial, destination)
            if digest_file(destination) != expected:
                raise OSError("Published Volume artifact failed validation")
            return destination
        except Exception as error:
            last_error = error
            try:
                partial.unlink(missing_ok=True)
            except OSError:
                pass
            if attempt < retries:
                time.sleep(2 ** (attempt - 1))
    raise OSError(f"Unable to publish {destination}") from last_error


def write_frame_atomic(frame, destination):
    destination = Path(destination)
    local = local_staging / f"{uuid.uuid4().hex}_{destination.name}"
    write_frame(frame, local)
    publish_local_artifact(local, destination)
    local.unlink(missing_ok=True)
    return destination


def write_json_atomic(payload, destination):
    destination = Path(destination)
    local = local_staging / f"{uuid.uuid4().hex}_{destination.name}"
    write_json(payload, local)
    publish_local_artifact(local, destination)
    local.unlink(missing_ok=True)
    return destination


progress_path = run_root / "run_progress.parquet"


def record_progress(stage, status, completed_batches=None, total_batches=None, details=None):
    row = pd.DataFrame(
        [
            {
                "stage": str(stage),
                "status": str(status),
                "completed_batches": completed_batches,
                "total_batches": total_batches,
                "details": details,
                "updated_unix_seconds": float(time.time()),
            }
        ]
    )
    if progress_path.exists() and progress_path.stat().st_size > 0:
        current = pd.read_parquet(progress_path)
        row = pd.concat([current, row], ignore_index=True, sort=False)
    row = row.sort_values("updated_unix_seconds").drop_duplicates(
        "stage", keep="last"
    )
    write_frame_atomic(row, progress_path)


write_json_atomic(
    {"run_id": run_root.name, "identity_sha256": identity_sha256},
    latest_pointer,
)

force_stages = {
    value.strip().lower() for value in widget("force_stages").split(",")
    if value.strip()
}
max_new_batches = int(widget("max_new_batches_per_stage"))
if max_new_batches < 0:
    raise ValueError("max_new_batches_per_stage cannot be negative")

config_public = {
    **identity,
    "run_id": run_root.name,
    "image_manifest_format": widget("image_manifest_format"),
    "cohort_format": widget("cohort_format"),
    "group_a_name": widget("group_a_name"),
    "group_b_name": widget("group_b_name"),
    "age_caliper_years": float(widget("age_caliper_years")),
    "match_ratio": int(widget("match_ratio")),
    "exact_match_columns": widget("exact_match_columns"),
    "manifest_batch_size": int(widget("manifest_batch_size")),
    "retfound_tensor_batch_size": int(widget("retfound_tensor_batch_size")),
    "segmentation_batch_size": int(widget("segmentation_batch_size")),
    "explainability_batch_size": int(widget("explainability_batch_size")),
    "run_segmentation": widget_bool("run_segmentation"),
    "run_explainability": widget_bool("run_explainability"),
    "run_targeted_occlusion": widget_bool("run_targeted_occlusion"),
    "device": widget("device"),
    "credentials_persisted": False,
}
write_json_atomic(config_public, run_root / "00_config" / "resolved_config.json")
print("Run directory:", run_root)
print("Resume identity:", identity_sha256[:12])

In [ ]:
SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".dcm"}


def read_parquet_compatibly(path):
    path = Path(normalized_fuse_path(path))
    if path.is_file():
        return pd.read_parquet(path)
    files = sorted(
        candidate for candidate in path.rglob("*.parquet")
        if candidate.is_file() and candidate.stat().st_size > 0
    )
    if not files:
        raise FileNotFoundError(f"No Parquet files found under {path}")
    frames = []
    for index, file_path in enumerate(files, start=1):
        frames.append(pd.read_parquet(file_path))
        if index % 100 == 0:
            print(f"Loaded {index:,}/{len(files):,} Parquet files", flush=True)
    output = pd.concat(frames, ignore_index=True, sort=False)
    output.attrs = {}
    return output


def read_table(path, table_format):
    path = normalized_fuse_path(path)
    if table_format == "delta":
        return spark.read.format("delta").load(path).toPandas()
    if table_format == "parquet":
        return read_parquet_compatibly(path)
    if table_format == "csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported table format: {table_format}")


def optional_table(path, table_format):
    return read_table(path, table_format) if str(path).strip() else None


def safe_archive_member(member):
    pure = PurePosixPath(member)
    if pure.is_absolute() or ".." in pure.parts:
        raise ValueError(f"Unsafe ZIP member path: {member}")
    return pure


def decode_dicom_to_jpeg(source, destination):
    import pydicom
    from pydicom.pixels import apply_voi_lut

    dataset = pydicom.dcmread(str(source))
    array = np.asarray(dataset.pixel_array)
    if array.ndim == 4:
        array = array[0]
    if array.ndim == 3 and array.shape[-1] not in (3, 4):
        array = array[0]
    if array.ndim == 2:
        try:
            array = np.asarray(apply_voi_lut(array, dataset))
        except Exception:
            pass
        finite = array[np.isfinite(array)]
        if not finite.size:
            raise ValueError("DICOM pixel array has no finite values")
        low, high = np.percentile(finite, [0.5, 99.5])
        if high <= low:
            low, high = float(finite.min()), float(finite.max())
        scaled = np.clip((array - low) / max(high - low, 1e-6), 0, 1)
        array = (scaled * 255).astype(np.uint8)
        if str(getattr(dataset, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            array = 255 - array
        image = Image.fromarray(array, mode="L").convert("RGB")
    else:
        array = array[..., :3]
        if array.dtype != np.uint8:
            finite = array[np.isfinite(array)]
            low, high = np.percentile(finite, [0.5, 99.5])
            array = (
                np.clip((array - low) / max(high - low, 1e-6), 0, 1) * 255
            ).astype(np.uint8)
        image = Image.fromarray(array).convert("RGB")
    destination = Path(destination)
    local = local_staging / f"{uuid.uuid4().hex}.jpg"
    image.save(local, format="JPEG", quality=95, subsampling=0)
    publish_local_artifact(local, destination)
    local.unlink(missing_ok=True)
    return destination


def parsed_path_value(pattern, value, group_name):
    if not pattern:
        return None
    match = re.search(pattern, str(value))
    return match.group(group_name) if match else None


source_type = widget("image_source_type")
image_source_paths = [
    Path(normalized_fuse_path(value))
    for value in source_path_text.split(";") if value.strip()
]
manifest_batch_size = int(widget("manifest_batch_size"))
input_root = run_root / "01_input_manifest"
input_batch_root = input_root / "batches"
input_batch_root.mkdir(parents=True, exist_ok=True)

if source_type == "manifest":
    raw_images = read_table(
        image_source_paths[0], widget("image_manifest_format")
    )
    configured_image_column = widget("image_path_column")
    if configured_image_column not in raw_images.columns:
        raise ValueError(
            f"Image manifest is missing {configured_image_column!r}"
        )
    raw_images = raw_images.rename(columns={configured_image_column: "image_path"})
    raw_images["source_reference"] = raw_images["image_path"].astype(str)
elif source_type == "directory":
    rows = []
    for directory in image_source_paths:
        if not directory.is_dir():
            raise FileNotFoundError(f"Image directory not found: {directory}")
        for path in directory.rglob("*"):
            if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS:
                rows.append({"image_path": str(path), "source_reference": str(path)})
    raw_images = pd.DataFrame(rows)
elif source_type == "zip":
    import pyzipper

    inventory_rows = []
    for archive in image_source_paths:
        if not archive.is_file():
            raise FileNotFoundError(f"ZIP archive not found: {archive}")
        with pyzipper.AESZipFile(archive) as handle:
            for info in handle.infolist():
                if info.is_dir():
                    continue
                member = safe_archive_member(info.filename)
                if Path(member.name).suffix.lower() not in SUPPORTED_IMAGE_EXTENSIONS:
                    continue
                source_reference = f"{archive}::{info.filename}"
                inventory_rows.append(
                    {
                        "archive_path": str(archive),
                        "archive_member": info.filename,
                        "source_reference": source_reference,
                        "image_key": stable_image_key(source_reference),
                        "encrypted": bool(info.flag_bits & 0x1),
                    }
                )
    archive_inventory = pd.DataFrame(inventory_rows).sort_values(
        "image_key", kind="stable"
    ).reset_index(drop=True)
    if archive_inventory.empty:
        raise ValueError("No supported image members were found in the ZIP input")
    write_frame_atomic(archive_inventory, input_root / "archive_inventory_private.parquet")
    archive_password = widget("archive_password")
    if archive_inventory["encrypted"].any() and not archive_password:
        raise ValueError(
            "At least one ZIP member is encrypted. Enter archive_password "
            "in the temporary widget before extraction."
        )
    extraction_paths = []
    new_batches = 0
    for start, stop in batch_ranges(len(archive_inventory), manifest_batch_size):
        batch_dir = input_batch_root / f"archive_{start:09d}_{stop:09d}"
        batch_path = batch_dir / "archive_extraction_private.parquet"
        expected = set(archive_inventory.iloc[start:stop]["image_key"])
        if batch_path.exists() and "input" not in force_stages:
            existing = pd.read_parquet(batch_path)
            if set(existing["image_key"].astype(str)) == expected:
                extraction_paths.append(batch_path)
                continue
        rows = []
        for record in archive_inventory.iloc[start:stop].itertuples():
            archive = Path(record.archive_path)
            member = safe_archive_member(record.archive_member)
            destination = input_root / "extracted" / stable_image_key(archive) / Path(*member.parts)
            destination.parent.mkdir(parents=True, exist_ok=True)
            error = None
            if not destination.is_file() or destination.stat().st_size < 1:
                try:
                    with pyzipper.AESZipFile(archive) as handle:
                        if archive_password:
                            handle.setpassword(archive_password.encode("utf-8"))
                        payload = handle.read(record.archive_member)
                    local = local_staging / f"{uuid.uuid4().hex}{destination.suffix}"
                    local.write_bytes(payload)
                    publish_local_artifact(local, destination)
                    local.unlink(missing_ok=True)
                except Exception as exception:
                    error = f"{type(exception).__name__}: {exception}"[:500]
            rows.append(
                {
                    **record._asdict(),
                    "image_path": str(destination),
                    "input_error": error,
                }
            )
        batch_frame = pd.DataFrame(rows)
        write_frame_atomic(batch_frame, batch_path)
        extraction_paths.append(batch_path)
        new_batches += 1
        print(
            f"[input ZIP] saved rows {start:,}:{stop:,}; "
            f"new batches={new_batches}", flush=True
        )
        if len(extraction_paths) % 10 == 0:
            record_progress(
                "zip_extraction", "running", len(extraction_paths),
                len(batch_ranges(len(archive_inventory), manifest_batch_size)),
            )
        if max_new_batches and new_batches >= max_new_batches:
            break
    archive_password = ""
    if len(extraction_paths) < len(batch_ranges(len(archive_inventory), manifest_batch_size)):
        record_progress(
            "zip_extraction", "checkpointed_incomplete", len(extraction_paths),
            len(batch_ranges(len(archive_inventory), manifest_batch_size)),
        )
        dbutils.notebook.exit(
            json.dumps(
                {
                    "status": "input_checkpointed_incomplete",
                    "completed_batches": len(extraction_paths),
                    "next_action": "Rerun this notebook; completed ZIP batches will be skipped.",
                    "run_root": str(run_root),
                }, indent=2
            )
        )
    raw_images = consolidate_batch_parquets(
        extraction_paths,
        key_column="image_key",
        expected_keys=archive_inventory["image_key"],
    )
else:
    raise ValueError(f"Unsupported image_source_type: {source_type}")

if raw_images.empty:
    raise ValueError("The image source produced no supported images")
raw_images["image_path"] = raw_images["image_path"].map(normalized_fuse_path)
if "source_reference" not in raw_images.columns:
    raw_images["source_reference"] = raw_images["image_path"].astype(str)
if "image_key" not in raw_images.columns:
    raw_images["image_key"] = raw_images["source_reference"].map(stable_image_key)
raw_images["image_key"] = raw_images["image_key"].astype(str)
if raw_images["image_key"].duplicated().any():
    raise ValueError("Input image references are not unique")
raw_images = raw_images.sort_values("image_key", kind="stable").reset_index(drop=True)

# Materialize DICOMs and validate ordinary paths in durable batches.
normalized_paths = []
new_batches = 0
materialize_root = input_root / "materialized_batches"
for start, stop in batch_ranges(len(raw_images), manifest_batch_size):
    batch_dir = materialize_root / f"batch_{start:09d}_{stop:09d}"
    batch_path = batch_dir / "normalized_image_manifest_private.parquet"
    expected = set(raw_images.iloc[start:stop]["image_key"])
    if batch_path.exists() and "input" not in force_stages:
        existing = pd.read_parquet(batch_path)
        if set(existing["image_key"].astype(str)) == expected:
            normalized_paths.append(batch_path)
            continue
    rows = []
    for record in raw_images.iloc[start:stop].to_dict("records"):
        row = dict(record)
        path = Path(str(row["image_path"]))
        row["original_image_path"] = str(path)
        row["input_error"] = row.get("input_error") or None
        if not row["input_error"]:
            try:
                if path.suffix.lower() == ".dcm":
                    destination = input_root / "dicom_rgb" / f"{row['image_key']}.jpg"
                    if not destination.exists() or destination.stat().st_size < 1:
                        decode_dicom_to_jpeg(path, destination)
                    row["image_path"] = str(destination)
                elif not path.is_file() or path.stat().st_size < 1:
                    raise FileNotFoundError(f"Image is absent or empty: {path}")
            except Exception as exception:
                row["input_error"] = f"{type(exception).__name__}: {exception}"[:500]
        rows.append(row)
    batch_frame = pd.DataFrame(rows)
    write_frame_atomic(batch_frame, batch_path)
    normalized_paths.append(batch_path)
    new_batches += 1
    print(f"[input] normalized rows {start:,}:{stop:,}", flush=True)
    if len(normalized_paths) % 10 == 0:
        record_progress(
            "input_normalization", "running", len(normalized_paths),
            len(batch_ranges(len(raw_images), manifest_batch_size)),
        )
    if max_new_batches and new_batches >= max_new_batches:
        break
if len(normalized_paths) < len(batch_ranges(len(raw_images), manifest_batch_size)):
    record_progress(
        "input_normalization", "checkpointed_incomplete", len(normalized_paths),
        len(batch_ranges(len(raw_images), manifest_batch_size)),
    )
    dbutils.notebook.exit(
        json.dumps(
            {
                "status": "input_checkpointed_incomplete",
                "completed_batches": len(normalized_paths),
                "next_action": "Rerun; completed image-normalization batches will be skipped.",
                "run_root": str(run_root),
            }, indent=2
        )
    )
images = consolidate_batch_parquets(
    normalized_paths,
    key_column="image_key",
    expected_keys=raw_images["image_key"],
)

participant_column = widget("participant_id_column")
eye_column = widget("eye_column")
visit_column = widget("visit_column")
sex_column = widget("sex_column")
device_column = widget("device_column")
if participant_column in images.columns:
    images["participant_id"] = images[participant_column].astype("string")
else:
    images["participant_id"] = images["source_reference"].map(
        lambda value: parsed_path_value(
            widget("participant_id_regex"), value, "participant_id"
        )
    ).astype("string")
if eye_column in images.columns:
    images["eye"] = images[eye_column].astype("string")
else:
    images["eye"] = images["source_reference"].map(
        lambda value: parsed_path_value(widget("eye_regex"), value, "eye")
    ).astype("string")
images["eye"] = images["eye"].str.lower().replace(
    {"os": "left", "od": "right", "l": "left", "r": "right"}
)
for configured, normalized in (
    (visit_column, "visit"),
    (sex_column, "sex"),
    (device_column, "device"),
):
    if configured in images.columns and normalized not in images.columns:
        images[normalized] = images[configured]

cohort_path = widget("cohort_path")
cohort = (
    read_table(cohort_path, widget("cohort_format"))
    if cohort_path else images.copy()
)
rename_map = {
    widget("participant_id_column"): "participant_id",
    widget("group_column"): "group_value",
    widget("age_column"): "age",
    widget("visit_column"): "visit",
    widget("sex_column"): "sex",
    widget("device_column"): "device",
}
rename_map = {
    source: target for source, target in rename_map.items()
    if source in cohort.columns and source != target
}
cohort = cohort.rename(columns=rename_map)
required_cohort = {"participant_id", "group_value", "age"}
missing = required_cohort - set(cohort.columns)
if missing:
    raise ValueError(f"Cohort input is missing required fields: {sorted(missing)}")
cohort["participant_id"] = cohort["participant_id"].astype(str).str.strip()
cohort["group_value"] = cohort["group_value"].astype(str).str.strip()
cohort["age"] = pd.to_numeric(cohort["age"], errors="coerce")
allowed_values = {widget("group_a_value"), widget("group_b_value")}
cohort = cohort[cohort["group_value"].isin(allowed_values)].copy()
cohort["group_label"] = (
    cohort["group_value"] == widget("group_b_value")
).astype(int)
if cohort.empty or set(cohort["group_label"]) != {0, 1}:
    raise ValueError("Cohort does not contain both configured groups")

join_columns = ["participant_id"]
if "visit" in images.columns and "visit" in cohort.columns:
    images["visit"] = images["visit"].astype(str).str.upper()
    cohort["visit"] = cohort["visit"].astype(str).str.upper()
    if images["visit"].notna().any():
        join_columns.append("visit")
retained = ["participant_id", "group_label", "group_value", "age"]
for column in ("visit", "sex", "device"):
    if column in cohort.columns:
        retained.append(column)
cohort_subset = cohort[retained].copy()
conflicting_groups = (
    cohort_subset.groupby(join_columns, dropna=False)["group_label"].nunique()
)
if (conflicting_groups > 1).any():
    raise ValueError(
        f"Cohort contains conflicting group values on join keys {join_columns}"
    )

def cohort_mode(series):
    values = series.dropna().astype(str)
    if values.empty:
        return np.nan
    return values.mode().sort_values().iloc[0]

cohort_aggregations = {
    "group_label": ("group_label", "first"),
    "group_value": ("group_value", "first"),
    "age": ("age", "median"),
}
for column in ("visit", "sex", "device"):
    if column in cohort_subset.columns and column not in join_columns:
        cohort_aggregations[column] = (column, cohort_mode)
cohort_link = cohort_subset.groupby(
    join_columns, as_index=False, dropna=False
).agg(**cohort_aggregations)
cohort_payload_columns = [
    column for column in retained
    if column not in join_columns and column in images.columns
]
images = images.drop(
    columns=cohort_payload_columns,
    errors="ignore",
).merge(cohort_link, on=join_columns, how="inner", validate="many_to_one")
images = images[
    images["participant_id"].notna() & images["age"].notna()
].copy()
if images.empty or set(images["group_label"].astype(int)) != {0, 1}:
    raise ValueError("No analyzable images remain in both configured groups")
write_frame_atomic(images, input_root / "normalized_image_manifest_private.parquet")
input_summary = (
    images.groupby("group_label", as_index=False)
    .agg(images=("image_key", "count"), participants=("participant_id", "nunique"))
)
display(input_summary)
print("Input rows with decode/path errors:", int(images["input_error"].notna().sum()))
record_progress("input_normalization", "complete", len(normalized_paths), len(normalized_paths))

In [ ]:
# Stage 2: resumable image quality assessment.
quality_root = run_root / "02_quality"
quality_batch_root = quality_root / "batches"
quality_batch_root.mkdir(parents=True, exist_ok=True)
quality_config = QualityConfig(
    output_size=256,
    model_input_size=224,
    save_preprocessed=widget_bool("save_preprocessed"),
)
external_quality = optional_table(
    widget("existing_quality_path"), widget("existing_artifact_format")
)
if external_quality is not None:
    if "image_key" not in external_quality.columns:
        if "original_image_path" in external_quality.columns:
            external_quality["image_key"] = external_quality["original_image_path"].map(stable_image_key)
        elif "image_path" in external_quality.columns:
            external_quality["image_key"] = external_quality["image_path"].map(stable_image_key)
        else:
            raise ValueError("Existing quality table needs image_key or image_path")
    external_quality["image_key"] = external_quality["image_key"].astype(str)
    external_quality = external_quality.drop_duplicates("image_key")

ordered_images = images.sort_values("image_key", kind="stable").reset_index(drop=True)
quality_paths = []
new_batches = 0
for batch_index, (start, stop) in enumerate(
    batch_ranges(len(ordered_images), manifest_batch_size), start=1
):
    batch_dir = quality_batch_root / f"batch_{start:09d}_{stop:09d}"
    batch_path = batch_dir / "fundus_quality_manifest.parquet"
    batch_input = ordered_images.iloc[start:stop].copy()
    expected = set(batch_input["image_key"].astype(str))
    if batch_path.exists() and "quality" not in force_stages:
        existing = pd.read_parquet(batch_path)
        if set(existing["image_key"].astype(str)) == expected and "quality_pass" in existing.columns:
            quality_paths.append(batch_path)
            print(f"[quality {batch_index}] resumed {len(existing)} rows", flush=True)
            continue
    reused_raw = (
        external_quality[external_quality["image_key"].isin(expected)].copy()
        if external_quality is not None else pd.DataFrame()
    )
    if not reused_raw.empty:
        quality_fields = [
            column for column in reused_raw.columns
            if column == "image_key" or column not in batch_input.columns
        ]
        reused = batch_input.merge(
            reused_raw[quality_fields], on="image_key", how="inner", validate="one_to_one"
        )
    else:
        reused = pd.DataFrame()
    missing_keys = expected - set(reused.get("image_key", pd.Series(dtype=str)).astype(str))
    missing_input = batch_input[batch_input["image_key"].isin(missing_keys)].copy()
    if len(missing_input):
        local_batch = local_staging / f"quality_{start}_{stop}_{uuid.uuid4().hex}"
        from fundus_retfound_pipeline import run_quality_pipeline
        calculated = run_quality_pipeline(missing_input, local_batch, quality_config)
        if quality_config.save_preprocessed:
            durable_processed = quality_root / "preprocessed_private"
            durable_processed.mkdir(parents=True, exist_ok=True)
            for row_index, row in calculated.iterrows():
                source_processed = row.get("processed_image_path")
                if source_processed and Path(str(source_processed)).is_file():
                    destination = durable_processed / f"{row['image_key']}.jpg"
                    publish_local_artifact(source_processed, destination)
                    calculated.at[row_index, "processed_image_path"] = str(destination)
        shutil.rmtree(local_batch, ignore_errors=True)
    else:
        calculated = pd.DataFrame()
    batch_output = pd.concat([reused, calculated], ignore_index=True, sort=False)
    if set(batch_output["image_key"].astype(str)) != expected:
        raise RuntimeError("Quality batch did not cover every expected image")
    write_frame_atomic(batch_output, batch_path)
    quality_paths.append(batch_path)
    new_batches += 1
    print(
        f"[quality {batch_index}] saved {len(batch_output)} rows; "
        f"pass={int(batch_output['quality_pass'].fillna(False).sum())}",
        flush=True,
    )
    if len(quality_paths) % 10 == 0:
        record_progress(
            "quality", "running", len(quality_paths),
            len(batch_ranges(len(ordered_images), manifest_batch_size)),
        )
    if max_new_batches and new_batches >= max_new_batches:
        break
expected_quality_batches = len(batch_ranges(len(ordered_images), manifest_batch_size))
if len(quality_paths) < expected_quality_batches:
    record_progress(
        "quality", "checkpointed_incomplete", len(quality_paths), expected_quality_batches
    )
    dbutils.notebook.exit(
        json.dumps(
            {
                "status": "quality_checkpointed_incomplete",
                "completed_batches": len(quality_paths),
                "total_batches": expected_quality_batches,
                "next_action": "Rerun this notebook; completed quality batches will be skipped.",
                "run_root": str(run_root),
            }, indent=2
        )
    )
quality = consolidate_batch_parquets(
    quality_paths,
    key_column="image_key",
    expected_keys=ordered_images["image_key"],
    required_columns=("quality_pass",),
)
write_frame_atomic(quality, quality_root / "fundus_quality_manifest_private.parquet")
quality_summary = (
    quality.groupby("group_label", as_index=False)
    .agg(
        images=("image_key", "count"),
        passing_images=("quality_pass", "sum"),
        participants=("participant_id", "nunique"),
    )
)
quality_summary["quality_pass_rate"] = (
    quality_summary["passing_images"] / quality_summary["images"]
)
write_frame_atomic(quality_summary, quality_root / "quality_summary.parquet")
display(quality_summary)
record_progress("quality", "complete", len(quality_paths), len(quality_paths))
if widget("stop_after_stage") == "quality":
    dbutils.notebook.exit(json.dumps({"status": "quality_complete", "run_root": str(run_root)}, indent=2))

In [ ]:
# Stage 3: build the quality-eligible participant cohort and age match.
matching_root = run_root / "03_age_matching"
matching_root.mkdir(parents=True, exist_ok=True)
passing = quality[quality["quality_pass"].fillna(False)].copy()
if passing.empty:
    raise ValueError("No images passed quality assessment")

exact_columns = [
    value.strip() for value in widget("exact_match_columns").split(",")
    if value.strip()
]
available_exact = [column for column in exact_columns if column in passing.columns]
unavailable_exact = sorted(set(exact_columns) - set(available_exact))
if unavailable_exact:
    print("Exact-match fields not available and omitted:", unavailable_exact)


def invariant_or_mode(series):
    values = series.dropna().astype(str)
    if values.empty:
        return np.nan
    modes = values.mode()
    return modes.sort_values().iloc[0]


aggregations = {
    "age": ("age", "median"),
    "group_label": ("group_label", "first"),
    "group_value": ("group_value", "first"),
    "quality_passing_images": ("image_key", "count"),
}
for column in set(available_exact + ["sex", "visit", "device"]):
    if column in passing.columns:
        aggregations[column] = (column, invariant_or_mode)
participants = passing.groupby("participant_id", as_index=False).agg(**aggregations)
group_invariance = passing.groupby("participant_id")["group_label"].nunique()
if (group_invariance > 1).any():
    raise ValueError("Some participants have conflicting group labels")
participants = participants.dropna(subset=["age", "group_label"]).copy()
participants["group_label"] = participants["group_label"].astype(int)
match_config = MatchConfig(
    ratio=int(widget("match_ratio")),
    caliper_years=float(widget("age_caliper_years")),
    exact_columns=tuple(available_exact),
)
pairs, match_audit, membership = match_participants(
    participants,
    match_config,
    id_column="participant_id",
    age_column="age",
    label_column="group_label",
)
write_frame_atomic(pairs, matching_root / "matched_pairs_private.parquet")
write_frame_atomic(match_audit, matching_root / "match_audit_private.parquet")
write_frame_atomic(membership, matching_root / "matched_membership_private.parquet")
if pairs.empty:
    aggregate_audit = (
        match_audit.groupby(["matched", "reason"], dropna=False)
        .size().rename("participants").reset_index()
    )
    write_frame_atomic(aggregate_audit, matching_root / "match_audit_summary.parquet")
    display(aggregate_audit)
    raise ValueError(
        "No participant matches were found. Review the nonidentifying audit, "
        "age overlap, exact-match fields, or caliper."
    )
matched_participants = membership.merge(
    participants, on=["participant_id", "group_label"], how="left", validate="one_to_one"
)
matched_images = passing.merge(
    membership[["participant_id", "match_set_id", "match_role"]],
    on="participant_id", how="inner", validate="many_to_one"
)
balance = matching_balance(participants, matched_participants)
unmatched_group_b = int((~match_audit["matched"]).sum())
matching_summary = {
    "eligible_group_a_participants": int((participants["group_label"] == 0).sum()),
    "eligible_group_b_participants": int((participants["group_label"] == 1).sum()),
    "matched_sets": int(pairs["match_set_id"].nunique()),
    "matched_group_a_participants": int((matched_participants["group_label"] == 0).sum()),
    "matched_group_b_participants": int((matched_participants["group_label"] == 1).sum()),
    "unmatched_group_b_participants": unmatched_group_b,
    "maximum_absolute_age_difference": float(pairs["absolute_age_difference_years"].max()),
    "exact_columns_used": available_exact,
}
write_json_atomic(matching_summary, matching_root / "matching_summary.json")
write_frame_atomic(balance, matching_root / "age_balance.parquet")
write_frame_atomic(matched_participants, matching_root / "matched_participants_private.parquet")
write_frame_atomic(matched_images, matching_root / "matched_images_private.parquet")
display(balance)
print(json.dumps(matching_summary, indent=2))
record_progress("matching", "complete", 1, 1)
if widget("stop_after_stage") == "matching":
    dbutils.notebook.exit(json.dumps({"status": "matching_complete", "run_root": str(run_root)}, indent=2))

In [ ]:
# Stage 4: load or calculate RETFound embeddings in durable manifest batches.
embedding_root = run_root / "04_retfound_embeddings"
embedding_batch_root = embedding_root / "batches"
embedding_batch_root.mkdir(parents=True, exist_ok=True)
external_embeddings = optional_table(
    widget("existing_embeddings_path"), widget("existing_artifact_format")
)
if external_embeddings is not None:
    if "embedding" not in external_embeddings.columns:
        raise ValueError("Existing embedding table is missing embedding")
    if "image_key" not in external_embeddings.columns:
        source_column = (
            "original_image_path" if "original_image_path" in external_embeddings.columns
            else "image_path"
        )
        external_embeddings["image_key"] = external_embeddings[source_column].map(stable_image_key)
    external_embeddings["image_key"] = external_embeddings["image_key"].astype(str)
    external_embeddings = external_embeddings.drop_duplicates("image_key")

ordered_embedding_input = matched_images.sort_values("image_key", kind="stable").reset_index(drop=True)
embedding_paths = []
new_batches = 0
retfound_model = None
retfound_device = None
resolved_checkpoint = None


def ensure_retfound_loaded():
    global retfound_model, retfound_device, resolved_checkpoint, resolved_retfound_repo
    if retfound_model is not None:
        return
    retfound_config = RETFoundConfig(
        repo_path=widget("retfound_repo") or None,
        checkpoint_path=widget("checkpoint_path") or None,
        allow_downloads=widget_bool("allow_checkpoint_download"),
        allow_repo_clone=widget_bool("allow_repo_clone"),
        device=widget("device"),
        batch_size=int(widget("retfound_tensor_batch_size")),
    )
    temporary_token = widget("hf_token")
    if temporary_token:
        os.environ["HF_TOKEN"] = temporary_token
    try:
        retfound_model, retfound_device, resolved_retfound_repo, resolved_checkpoint = (
            load_retfound_model(retfound_config)
        )
    finally:
        os.environ.pop("HF_TOKEN", None)
        temporary_token = ""


for batch_index, (start, stop) in enumerate(
    batch_ranges(len(ordered_embedding_input), manifest_batch_size), start=1
):
    batch_dir = embedding_batch_root / f"batch_{start:09d}_{stop:09d}"
    batch_path = batch_dir / "embedding_status_private.parquet"
    batch_input = ordered_embedding_input.iloc[start:stop].copy()
    expected = set(batch_input["image_key"].astype(str))
    if batch_path.exists() and "embeddings" not in force_stages:
        existing = pd.read_parquet(batch_path)
        if set(existing["image_key"].astype(str)) == expected and "embedding_ok" in existing.columns:
            embedding_paths.append(batch_path)
            print(f"[embedding {batch_index}] resumed {len(existing)} rows", flush=True)
            continue
    reused_raw = (
        external_embeddings[external_embeddings["image_key"].isin(expected)].copy()
        if external_embeddings is not None else pd.DataFrame()
    )
    if not reused_raw.empty:
        embedding_fields = [
            column for column in reused_raw.columns
            if column == "image_key" or column not in batch_input.columns
        ]
        reused = batch_input.merge(
            reused_raw[embedding_fields], on="image_key", how="inner", validate="one_to_one"
        )
    else:
        reused = pd.DataFrame()
    if not reused.empty:
        reused["embedding_ok"] = True
        reused["embedding_error"] = None
    missing_keys = expected - set(reused.get("image_key", pd.Series(dtype=str)).astype(str))
    missing_input = batch_input[batch_input["image_key"].isin(missing_keys)].copy()
    calculated = pd.DataFrame()
    failed = pd.DataFrame()
    if len(missing_input):
        ensure_retfound_loaded()
        local_batch = local_staging / f"embedding_{start}_{stop}_{uuid.uuid4().hex}"
        retfound_config = RETFoundConfig(
            repo_path=str(resolved_retfound_repo),
            checkpoint_path=str(resolved_checkpoint),
            allow_downloads=False,
            allow_repo_clone=False,
            device=retfound_device,
            batch_size=int(widget("retfound_tensor_batch_size")),
        )
        try:
            calculated = extract_retfound_embeddings(
                missing_input,
                local_batch,
                retfound_config,
                quality_config,
                model=retfound_model,
                device=retfound_device,
                checkpoint_path=resolved_checkpoint,
                force=True,
            )
            calculated["embedding_ok"] = True
            calculated["embedding_error"] = None
            failure_path = local_batch / "retfound_embedding_failures.csv"
            if failure_path.exists() and failure_path.stat().st_size > 0:
                failed = pd.read_csv(failure_path)
        except RuntimeError as error:
            if "Every image failed" not in str(error):
                raise
            failed = missing_input[["image_key", "image_path"]].copy()
            failed["error"] = str(error)
        finally:
            shutil.rmtree(local_batch, ignore_errors=True)
    success_keys = set(calculated.get("image_key", pd.Series(dtype=str)).astype(str))
    failure_lookup = {}
    if not failed.empty and "image_path" in failed.columns:
        failure_lookup = dict(zip(failed["image_path"].astype(str), failed.get("error", "failed")))
    status_rows = []
    successful = pd.concat([reused, calculated], ignore_index=True, sort=False)
    successful_lookup = {
        str(row["image_key"]): row for row in successful.to_dict("records")
    }
    for base in batch_input.to_dict("records"):
        key = str(base["image_key"])
        if key in successful_lookup:
            row = {**base, **successful_lookup[key]}
            row["embedding_ok"] = True
            row["embedding_error"] = None
        else:
            row = dict(base)
            row["embedding"] = None
            row["embedding_dim"] = None
            row["embedding_ok"] = False
            row["embedding_error"] = str(failure_lookup.get(str(base["image_path"]), "embedding_failed"))[:500]
        status_rows.append(row)
    batch_output = pd.DataFrame(status_rows)
    write_frame_atomic(batch_output, batch_path)
    embedding_paths.append(batch_path)
    new_batches += 1
    print(
        f"[embedding {batch_index}] saved {len(batch_output)} rows; "
        f"success={int(batch_output['embedding_ok'].sum())}", flush=True
    )
    if len(embedding_paths) % 10 == 0:
        record_progress(
            "embeddings", "running", len(embedding_paths),
            len(batch_ranges(len(ordered_embedding_input), manifest_batch_size)),
        )
    if max_new_batches and new_batches >= max_new_batches:
        break
expected_embedding_batches = len(batch_ranges(len(ordered_embedding_input), manifest_batch_size))
if len(embedding_paths) < expected_embedding_batches:
    record_progress(
        "embeddings", "checkpointed_incomplete", len(embedding_paths), expected_embedding_batches
    )
    dbutils.notebook.exit(
        json.dumps(
            {
                "status": "embedding_checkpointed_incomplete",
                "completed_batches": len(embedding_paths),
                "total_batches": expected_embedding_batches,
                "next_action": "Rerun on GPU compute; completed embedding batches will be skipped.",
                "run_root": str(run_root),
            }, indent=2
        )
    )
embedding_status = consolidate_batch_parquets(
    embedding_paths,
    key_column="image_key",
    expected_keys=ordered_embedding_input["image_key"],
    required_columns=("embedding_ok",),
)
embeddings = embedding_status[embedding_status["embedding_ok"].fillna(False)].copy()
dimensions = sorted(
    {
        int(np.asarray(vector).reshape(-1).size)
        for vector in embeddings["embedding"]
    }
)
if dimensions != [1024]:
    raise ValueError(f"Expected 1,024-dimensional RETFound embeddings, found {dimensions}")
participants_without_embeddings = set(matched_participants["participant_id"].astype(str)) - set(
    embeddings["participant_id"].astype(str)
)
if participants_without_embeddings:
    raise ValueError(
        f"{len(participants_without_embeddings):,} matched participants have no successful "
        "embeddings. Review embedding_status_private.parquet before inference."
    )
write_frame_atomic(embedding_status, embedding_root / "embedding_status_private.parquet")
write_frame_atomic(embeddings, embedding_root / "retfound_embeddings_private.parquet")
print("Successful embeddings:", len(embeddings))
record_progress("embeddings", "complete", len(embedding_paths), len(embedding_paths))
if widget("stop_after_stage") == "embeddings":
    dbutils.notebook.exit(json.dumps({"status": "embeddings_complete", "run_root": str(run_root)}, indent=2))

In [ ]:
# Stage 5: resumable vessel segmentation and disc/fovea localization.
# The encoder is reloaded for explainability later. Releasing it here
# prevents simultaneous RETFound and segmentation models exhausting a T4.
retfound_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
segmentation_root = run_root / "05_segmentations"
segmentation_batch_root = segmentation_root / "batches"
segmentation_mask_root = segmentation_root / "masks_private"
segmentation_overlay_root = segmentation_root / "overlays_private"
for path in (segmentation_batch_root, segmentation_mask_root, segmentation_overlay_root):
    path.mkdir(parents=True, exist_ok=True)


def save_segmentation_overlay(rgb, masks, destination):
    figure, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb)
    axes[0].set_title("Quality-normalized fundus")
    axes[1].imshow(rgb)
    axes[1].imshow(masks["vessels"], cmap="Reds", alpha=0.65)
    axes[1].set_title("Vessel segmentation")
    axes[2].imshow(rgb)
    for name, color in (
        ("optic_disc_roi", "cyan"),
        ("optic_disc_plus_peripapillary", "yellow"),
        ("fovea_roi", "lime"),
    ):
        axes[2].contour(masks[name], levels=[0.5], colors=[color], linewidths=1)
    axes[2].set_title("Disc/peripapillary/foveal ROIs")
    for axis in axes:
        axis.axis("off")
    figure.tight_layout()
    figure.savefig(destination, dpi=160, bbox_inches="tight")
    plt.close(figure)


segmentation_status = pd.DataFrame()
if widget_bool("run_segmentation"):
    fit_cache = output_base / "_model_cache" / "fundus_image_toolbox"
    fit_cache.mkdir(parents=True, exist_ok=True)
    os.environ["FIT_CACHE_DIR"] = str(fit_cache)
    os.environ["TORCH_HOME"] = str(fit_cache / "torch")
    import fundus_image_toolbox as fit
    anatomy_device = (
        "cuda:0" if widget("device") in {"auto", "cuda"} and torch.cuda.is_available()
        else "cpu"
    )
    external_segmentations = optional_table(
        widget("existing_segmentations_path"), widget("existing_artifact_format")
    )
    if external_segmentations is not None:
        if "image_key" not in external_segmentations.columns:
            external_segmentations["image_key"] = external_segmentations["image_path"].map(stable_image_key)
        external_segmentations["image_key"] = external_segmentations["image_key"].astype(str)
        external_segmentations = external_segmentations.drop_duplicates("image_key")
    ordered_segmentation = embeddings.sort_values("image_key", kind="stable").reset_index(drop=True)
    segmentation_paths = []
    new_batches = 0
    landmark_model = None
    vessel_ensemble = None

    def ensure_anatomy_models_loaded():
        global landmark_model, vessel_ensemble
        if landmark_model is None:
            print("Loading Fundus Image Toolbox landmark and vessel models...", flush=True)
            landmark_model, _ = fit.load_fovea_od_model(
                device=anatomy_device, cache_dir=str(fit_cache)
            )
            vessel_ensemble = fit.load_segmentation_ensemble(
                device=anatomy_device, cache_dir=str(fit_cache)
            )

    segmentation_batch_size = int(widget("segmentation_batch_size"))
    for batch_index, (start, stop) in enumerate(
        batch_ranges(len(ordered_segmentation), segmentation_batch_size), start=1
    ):
        batch_dir = segmentation_batch_root / f"batch_{start:09d}_{stop:09d}"
        batch_path = batch_dir / "segmentation_status_private.parquet"
        batch_input = ordered_segmentation.iloc[start:stop].copy()
        expected = set(batch_input["image_key"].astype(str))
        if batch_path.exists() and "segmentation" not in force_stages:
            existing = pd.read_parquet(batch_path)
            if set(existing["image_key"].astype(str)) == expected and "segmentation_ok" in existing.columns:
                segmentation_paths.append(batch_path)
                print(f"[segmentation {batch_index}] resumed {len(existing)} rows", flush=True)
                continue
        reused = (
            external_segmentations[external_segmentations["image_key"].isin(expected)].copy()
            if external_segmentations is not None else pd.DataFrame()
        )
        if not reused.empty:
            reused["segmentation_ok"] = reused.get("anatomy_valid", True)
            reused["segmentation_error"] = None
        reused_keys = set(reused.get("image_key", pd.Series(dtype=str)).astype(str))
        missing = batch_input[~batch_input["image_key"].isin(reused_keys)].copy()
        calculated_rows = []
        if len(missing):
            ensure_anatomy_models_loaded()
            processed_records = []
            processed_images = []
            for record in missing.to_dict("records"):
                try:
                    processed = preprocess_fundus(record["image_path"], quality_config)
                    processed_records.append(record)
                    processed_images.append(np.asarray(processed.image.convert("RGB")))
                except Exception as exception:
                    calculated_rows.append(
                        {**record, "segmentation_ok": False, "anatomy_valid": False,
                         "segmentation_error": f"{type(exception).__name__}: {exception}"[:500]}
                    )
            if processed_images:
                coordinates = np.asarray(
                    landmark_model.predict(processed_images), dtype=float
                ).reshape(-1, 4)
                vessel_predictions = np.asarray(
                    fit.ensemble_predict_segmentation(
                        vessel_ensemble, processed_images, device=anatomy_device,
                        size=(512, 512), threshold=0.5,
                    ), dtype=float
                )
                if vessel_predictions.ndim == 2:
                    vessel_predictions = vessel_predictions[None, ...]
                for position, record in enumerate(processed_records):
                    try:
                        rgb = processed_images[position]
                        vessel_mask = vessel_predictions[position]
                        if vessel_mask.shape != rgb.shape[:2]:
                            vessel_mask = np.asarray(
                                Image.fromarray(vessel_mask.astype(np.float32)).resize(
                                    (rgb.shape[1], rgb.shape[0]), Image.Resampling.BILINEAR
                                )
                            )
                        retina = fundus_physiology_proxies(rgb)["retina"]
                        masks, metadata = build_anatomic_masks(
                            rgb.shape[:2], coordinates[position], vessel_mask, retina
                        )
                        key = str(record["image_key"])
                        mask_destination = segmentation_mask_root / f"{key}_anatomic_masks.npz"
                        local_mask = local_staging / f"{uuid.uuid4().hex}.npz"
                        np.savez_compressed(
                            local_mask,
                            **{name: value.astype(np.uint8) for name, value in masks.items()},
                        )
                        publish_local_artifact(local_mask, mask_destination)
                        local_mask.unlink(missing_ok=True)
                        overlay_destination = segmentation_overlay_root / f"{key}_segmentation.png"
                        local_overlay = local_staging / f"{uuid.uuid4().hex}.png"
                        save_segmentation_overlay(rgb, masks, local_overlay)
                        publish_local_artifact(local_overlay, overlay_destination)
                        local_overlay.unlink(missing_ok=True)
                        calculated_rows.append(
                            {
                                **record,
                                **metadata,
                                "segmentation_ok": bool(metadata["anatomy_valid"]),
                                "segmentation_error": None,
                                "mask_path": str(mask_destination),
                                "segmentation_overlay_path": str(overlay_destination),
                                "fit_version": fit.__version__,
                                "fit_source_commit": "d7757e28fbf639856b53cfe00019f605af8c1f17",
                            }
                        )
                    except Exception as exception:
                        calculated_rows.append(
                            {**record, "segmentation_ok": False, "anatomy_valid": False,
                             "segmentation_error": f"{type(exception).__name__}: {exception}"[:500]}
                        )
        batch_output = pd.concat(
            [reused, pd.DataFrame(calculated_rows)], ignore_index=True, sort=False
        )
        if set(batch_output["image_key"].astype(str)) != expected:
            raise RuntimeError("Segmentation batch did not cover every expected image")
        write_frame_atomic(batch_output, batch_path)
        segmentation_paths.append(batch_path)
        new_batches += 1
        print(
            f"[segmentation {batch_index}] saved {len(batch_output)} rows; "
            f"valid={int(batch_output['segmentation_ok'].fillna(False).sum())}", flush=True
        )
        if len(segmentation_paths) % 25 == 0:
            record_progress(
                "segmentation", "running", len(segmentation_paths),
                len(batch_ranges(len(ordered_segmentation), segmentation_batch_size)),
            )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if max_new_batches and new_batches >= max_new_batches:
            break
    expected_segmentation_batches = len(batch_ranges(len(ordered_segmentation), segmentation_batch_size))
    if len(segmentation_paths) < expected_segmentation_batches:
        record_progress(
            "segmentation", "checkpointed_incomplete",
            len(segmentation_paths), expected_segmentation_batches,
        )
        dbutils.notebook.exit(
            json.dumps(
                {
                    "status": "segmentation_checkpointed_incomplete",
                    "completed_batches": len(segmentation_paths),
                    "total_batches": expected_segmentation_batches,
                    "next_action": "Rerun; completed segmentation batches will be skipped.",
                    "run_root": str(run_root),
                }, indent=2
            )
        )
    segmentation_status = consolidate_batch_parquets(
        segmentation_paths,
        key_column="image_key",
        expected_keys=ordered_segmentation["image_key"],
        required_columns=("segmentation_ok",),
    )
    write_frame_atomic(segmentation_status, segmentation_root / "segmentation_status_private.parquet")
    print("Valid segmentations:", int(segmentation_status["segmentation_ok"].fillna(False).sum()))
    record_progress("segmentation", "complete", len(segmentation_paths), len(segmentation_paths))
else:
    print("Segmentation disabled by configuration")
    record_progress("segmentation", "skipped", 0, 0)
if widget("stop_after_stage") == "segmentation":
    dbutils.notebook.exit(json.dumps({"status": "segmentation_stage_complete", "run_root": str(run_root)}, indent=2))

In [ ]:
# Stage 6: participant aggregation and match-set-preserving nested CV.
model_root = run_root / "06_participant_model"
model_root.mkdir(parents=True, exist_ok=True)
participant_embeddings = aggregate_participant_embeddings(
    embeddings,
    expected_dim=1024,
    carry_columns=("group_label", "age", "match_set_id"),
)
participant_embeddings["group_label"] = participant_embeddings["group_label"].astype(int)
write_frame_atomic(
    participant_embeddings,
    model_root / "participant_embeddings_private.parquet",
)
predictions, fold_heads, final_head = fit_grouped_oof_classifier(
    participant_embeddings,
    folds=5,
    inner_folds=4,
    expected_dim=1024,
    c_grid=(0.001, 0.01, 0.1, 1.0),
    group_column="match_set_id",
)
model_metrics = prediction_metrics(
    predictions["group_label"],
    predictions["group_b_probability_oof"],
    bootstrap_repetitions=2000,
)
predictions = predictions.merge(
    participant_embeddings.drop(columns=["embedding"]),
    on=["participant_id", "group_label", "match_set_id"],
    how="left", validate="one_to_one",
)
write_frame_atomic(predictions, model_root / "participant_oof_predictions_private.parquet")
write_json_atomic(model_metrics, model_root / "participant_oof_metrics.json")
for payload, filename in (
    (fold_heads, "oof_fold_heads.joblib"),
    (final_head, "frozen_final_head.joblib"),
):
    local_model = local_staging / f"{uuid.uuid4().hex}_{filename}"
    joblib.dump(payload, local_model)
    publish_local_artifact(local_model, model_root / filename)
    local_model.unlink(missing_ok=True)
image_predictions = embeddings.merge(
    predictions[
        ["participant_id", "fold", "classifier_logit_oof", "group_b_probability_oof"]
    ],
    on="participant_id", how="left", validate="many_to_one",
)
write_frame_atomic(image_predictions, model_root / "image_model_manifest_private.parquet")
print(json.dumps(model_metrics, indent=2))
record_progress("participant_model", "complete", 1, 1)

# Device/source overlap diagnostic. This never silently harmonizes data.
domain_rows = []
perfect_device_group_confounding = False
if "device" in matched_images.columns and matched_images["device"].notna().any():
    participant_device = (
        matched_images.groupby("participant_id")["device"].agg(invariant_or_mode)
        .rename("device").reset_index()
    )
    domain = predictions.merge(participant_device, on="participant_id", how="left")
    contingency = pd.crosstab(domain["device"].fillna("missing"), domain["group_label"])
    for device_name, row in contingency.iterrows():
        domain_rows.append(
            {
                "device": str(device_name),
                "group_a_participants": int(row.get(0, 0)),
                "group_b_participants": int(row.get(1, 0)),
                "contains_both_groups": bool(row.get(0, 0) > 0 and row.get(1, 0) > 0),
            }
        )
    perfect_device_group_confounding = not any(
        row["contains_both_groups"] for row in domain_rows
    )
    within_device_rows = []
    for device_name, device_frame in domain.groupby("device", dropna=False):
        class_counts = device_frame["group_label"].value_counts()
        if set(class_counts.index) != {0, 1} or int(class_counts.min()) < 5:
            continue
        device_metrics = prediction_metrics(
            device_frame["group_label"],
            device_frame["group_b_probability_oof"],
            bootstrap_repetitions=500,
            random_state=20260819 + len(within_device_rows),
        )
        within_device_rows.append(
            {"device": str(device_name), **device_metrics}
        )
    if within_device_rows:
        write_frame_atomic(
            pd.DataFrame(within_device_rows),
            model_root / "within_device_oof_sensitivity.parquet",
        )
domain_diagnostic = pd.DataFrame(domain_rows)
if len(domain_diagnostic):
    write_frame_atomic(domain_diagnostic, model_root / "device_group_overlap.parquet")
    display(domain_diagnostic)
write_json_atomic(
    {
        "device_field_available": bool(len(domain_rows)),
        "perfect_device_group_confounding": perfect_device_group_confounding,
        "disease_specific_inference_identifiable_within_device": (
            bool(len(domain_rows)) and not perfect_device_group_confounding
        ),
    },
    model_root / "domain_shift_diagnostic.json",
)
if perfect_device_group_confounding:
    print(
        "WARNING: no device contains both groups. Group and device effects are not "
        "separately identifiable; treat classifier results as domain-plus-group discrimination."
    )
if widget("stop_after_stage") == "model":
    dbutils.notebook.exit(json.dumps({"status": "model_complete", "run_root": str(run_root)}, indent=2))

In [ ]:
# Stage 7: exact patch attribution, anatomic quantification, and optional occlusion.
explanation_root = run_root / "07_explainability"
explanation_batch_root = explanation_root / "batches"
explanation_map_root = explanation_root / "maps_private"
explanation_overlay_root = explanation_root / "overlays_private"
for path in (explanation_batch_root, explanation_map_root, explanation_overlay_root):
    path.mkdir(parents=True, exist_ok=True)


def resize_mask(mask, shape):
    image = Image.fromarray(np.asarray(mask, dtype=np.uint8) * 255)
    return np.asarray(image.resize((shape[1], shape[0]), Image.Resampling.NEAREST)) > 0


def save_explainability_overlay(rgb, grid, masks, destination):
    heat = np.asarray(
        Image.fromarray(np.asarray(grid, dtype=np.float32), mode="F").resize(
            (rgb.shape[1], rgb.shape[0]), Image.Resampling.BILINEAR
        ), dtype=float
    )
    limit = max(float(np.quantile(np.abs(heat), 0.99)), 1e-8)
    figure, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb)
    axes[0].set_title("RETFound input")
    axes[1].imshow(heat, cmap="coolwarm", vmin=-limit, vmax=limit)
    axes[1].set_title("Exact group-B contribution")
    axes[2].imshow(rgb)
    axes[2].imshow(heat, cmap="coolwarm", vmin=-limit, vmax=limit, alpha=0.45)
    if masks:
        for name, color in (
            ("optic_disc_roi", "cyan"),
            ("optic_disc_plus_peripapillary", "yellow"),
            ("fovea_roi", "lime"),
            ("vessels", "red"),
        ):
            if name in masks:
                axes[2].contour(masks[name], levels=[0.5], colors=[color], linewidths=0.8)
    axes[2].set_title("Attribution and anatomy")
    for axis in axes:
        axis.axis("off")
    figure.tight_layout()
    figure.savefig(destination, dpi=170, bbox_inches="tight")
    plt.close(figure)


explanation_status = pd.DataFrame()
if widget_bool("run_explainability"):
    ensure_retfound_loaded()
    selected_images = image_predictions.sort_values(
        ["group_label", "participant_id", "image_key"], kind="stable"
    ).reset_index(drop=True)
    maximum_images = int(widget("maximum_explainability_images"))
    if maximum_images > 0:
        per_group = max(1, maximum_images // 2)
        selected_images = pd.concat(
            [group.head(per_group) for _, group in selected_images.groupby("group_label")],
            ignore_index=True,
        ).head(maximum_images)
    if len(segmentation_status):
        segmentation_link = segmentation_status.drop(
            columns=[column for column in selected_images.columns if column != "image_key"],
            errors="ignore",
        )
        selected_images = selected_images.merge(
            segmentation_link, on="image_key", how="left", validate="one_to_one"
        )
    selected_images = selected_images.reset_index(drop=True)
    explanation_batch_size = int(widget("explainability_batch_size"))
    explanation_paths = []
    new_batches = 0
    for batch_index, (start, stop) in enumerate(
        batch_ranges(len(selected_images), explanation_batch_size), start=1
    ):
        batch_dir = explanation_batch_root / f"batch_{start:09d}_{stop:09d}"
        batch_path = batch_dir / "explainability_status_private.parquet"
        batch_input = selected_images.iloc[start:stop].copy()
        expected = set(batch_input["image_key"].astype(str))
        if batch_path.exists() and "explainability" not in force_stages:
            existing = pd.read_parquet(batch_path)
            if set(existing["image_key"].astype(str)) == expected and "explainability_ok" in existing.columns:
                explanation_paths.append(batch_path)
                print(f"[explainability {batch_index}] resumed {len(existing)} rows", flush=True)
                continue
        rows = []
        for record in batch_input.to_dict("records"):
            try:
                fold = int(record["fold"])
                head = fold_heads[fold]
                attribution = exact_linear_patch_contributions(
                    retfound_model,
                    head["coefficients"],
                    head["intercept"],
                    record["image_path"],
                    retfound_device,
                    quality_config,
                )
                if attribution["reconstruction_error"] > 1e-4:
                    raise RuntimeError(
                        f"Exact attribution reconstruction error={attribution['reconstruction_error']}"
                    )
                key = str(record["image_key"])
                map_destination = explanation_map_root / f"{key}_exact_attribution.npz"
                local_map = local_staging / f"{uuid.uuid4().hex}.npz"
                np.savez_compressed(
                    local_map,
                    grid=attribution["grid"],
                    variable_grid=attribution["variable_grid"],
                )
                publish_local_artifact(local_map, map_destination)
                local_map.unlink(missing_ok=True)
                processed = preprocess_fundus(record["image_path"], quality_config)
                rgb = np.asarray(processed.image.convert("RGB"))
                retina = fundus_physiology_proxies(rgb)["retina"]
                masks = {}
                region_metrics = {}
                if record.get("mask_path") and Path(str(record["mask_path"])).exists():
                    with np.load(record["mask_path"]) as saved_masks:
                        masks = {name: np.asarray(saved_masks[name], dtype=bool) for name in saved_masks.files}
                    region_metrics = attribution_region_metrics(
                        attribution["variable_grid"], retina, masks
                    )
                occlusion = {}
                if widget_bool("run_targeted_occlusion") and masks:
                    model_array = prepare_model_input(record["image_path"], quality_config)
                    model_retina = resize_mask(retina, model_array.shape[:2])

                    def score(mask=None):
                        altered = model_array.copy()
                        if mask is not None:
                            altered[mask] = 0.0
                        return linear_head_score_from_array(
                            retfound_model, head["coefficients"], head["intercept"],
                            altered, retfound_device,
                        )

                    baseline = score()
                    excluded = np.zeros(model_array.shape[:2], dtype=bool)
                    for excluded_name in ("optic_disc_plus_peripapillary", "fovea_roi", "vessels"):
                        if excluded_name in masks:
                            excluded |= resize_mask(masks[excluded_name], model_array.shape[:2])
                    for region_name in (
                        "optic_disc_roi", "optic_disc_plus_peripapillary",
                        "fovea_roi", "vessels_elsewhere",
                    ):
                        if region_name not in masks:
                            continue
                        region = resize_mask(masks[region_name], model_array.shape[:2]) & model_retina
                        if not region.any():
                            continue
                        region_drop = baseline - score(region)
                        controls = sample_equal_area_control_masks(
                            model_retina, excluded,
                            target_area=int(region.sum()),
                            n_masks=int(widget("occlusion_control_masks")),
                            random_state=20260819 + int(key[:8], 16),
                        )
                        control_drops = np.asarray(
                            [baseline - score(control) for control in controls], dtype=float
                        )
                        occlusion[f"{region_name}_occlusion_logit_drop"] = float(region_drop)
                        occlusion[f"{region_name}_specific_occlusion_drop"] = (
                            float(region_drop - np.median(control_drops))
                            if len(control_drops) else np.nan
                        )
                        occlusion[f"{region_name}_n_control_masks"] = int(len(control_drops))
                overlay_destination = explanation_overlay_root / f"{key}_explainability.png"
                local_overlay = local_staging / f"{uuid.uuid4().hex}.png"
                save_explainability_overlay(
                    rgb, attribution["variable_grid"], masks, local_overlay
                )
                publish_local_artifact(local_overlay, overlay_destination)
                local_overlay.unlink(missing_ok=True)
                rows.append(
                    {
                        **record,
                        "explainability_ok": True,
                        "explainability_error": None,
                        "attribution_map_path": str(map_destination),
                        "explainability_overlay_path": str(overlay_destination),
                        "image_logit_from_exact_attribution": float(attribution["prediction_from_feature"]),
                        "attribution_reconstruction_error": float(attribution["reconstruction_error"]),
                        **region_metrics,
                        **occlusion,
                    }
                )
            except Exception as exception:
                rows.append(
                    {
                        **record,
                        "explainability_ok": False,
                        "explainability_error": f"{type(exception).__name__}: {exception}"[:500],
                    }
                )
        batch_output = pd.DataFrame(rows)
        write_frame_atomic(batch_output, batch_path)
        explanation_paths.append(batch_path)
        new_batches += 1
        print(
            f"[explainability {batch_index}] saved {len(batch_output)} rows; "
            f"success={int(batch_output['explainability_ok'].sum())}", flush=True
        )
        if len(explanation_paths) % 10 == 0:
            record_progress(
                "explainability", "running", len(explanation_paths),
                len(batch_ranges(len(selected_images), explanation_batch_size)),
            )
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if max_new_batches and new_batches >= max_new_batches:
            break
    expected_explanation_batches = len(batch_ranges(len(selected_images), explanation_batch_size))
    if len(explanation_paths) < expected_explanation_batches:
        record_progress(
            "explainability", "checkpointed_incomplete",
            len(explanation_paths), expected_explanation_batches,
        )
        dbutils.notebook.exit(
            json.dumps(
                {
                    "status": "explainability_checkpointed_incomplete",
                    "completed_batches": len(explanation_paths),
                    "total_batches": expected_explanation_batches,
                    "next_action": "Rerun; completed explainability batches will be skipped.",
                    "run_root": str(run_root),
                }, indent=2
            )
        )
    explanation_status = consolidate_batch_parquets(
        explanation_paths,
        key_column="image_key",
        expected_keys=selected_images["image_key"],
        required_columns=("explainability_ok",),
    )
    write_frame_atomic(
        explanation_status,
        explanation_root / "explainability_status_private.parquet",
    )
    print("Successful explanations:", int(explanation_status["explainability_ok"].sum()))
    record_progress("explainability", "complete", len(explanation_paths), len(explanation_paths))
else:
    print("Explainability disabled by configuration")
    record_progress("explainability", "skipped", 0, 0)
if widget("stop_after_stage") == "explainability":
    dbutils.notebook.exit(json.dumps({"status": "explainability_complete", "run_root": str(run_root)}, indent=2))

In [ ]:
# Stage 8: aggregate statistics and publication-oriented figures.
statistics_root = run_root / "08_statistics"
figure_root = run_root / "09_figures"
statistics_root.mkdir(parents=True, exist_ok=True)
figure_root.mkdir(parents=True, exist_ok=True)


def save_figure_atomic(figure, filename):
    destination = figure_root / filename
    local_figure = local_staging / f"{uuid.uuid4().hex}_{filename}"
    figure.savefig(local_figure, dpi=220, bbox_inches="tight")
    publish_local_artifact(local_figure, destination)
    local_figure.unlink(missing_ok=True)
    return destination

# Cohort derivation contains counts only.
derivation = pd.DataFrame(
    [
        {"stage": "Input participants", "participants": images["participant_id"].nunique()},
        {"stage": "Quality-eligible participants", "participants": participants["participant_id"].nunique()},
        {"stage": "Matched participants", "participants": matched_participants["participant_id"].nunique()},
        {"stage": "Embedded participants", "participants": participant_embeddings["participant_id"].nunique()},
    ]
)
write_frame_atomic(derivation, statistics_root / "cohort_derivation.parquet")
write_frame_atomic(quality_summary, statistics_root / "quality_summary.parquet")
write_frame_atomic(balance, statistics_root / "matching_age_balance.parquet")

roi_statistics = pd.DataFrame()
participant_anatomy = pd.DataFrame()
if len(explanation_status):
    successful = explanation_status[explanation_status["explainability_ok"].fillna(False)].copy()
    metric_columns = [
        column for column in successful.columns
        if column.endswith("_positive_enrichment")
        or column.endswith("_absolute_enrichment")
        or column.endswith("_specific_occlusion_drop")
    ]
    if metric_columns:
        participant_anatomy = (
            successful.groupby(
                ["participant_id", "group_label", "match_set_id"], as_index=False
            )[metric_columns].mean()
        )
        write_frame_atomic(
            participant_anatomy,
            statistics_root / "participant_anatomic_metrics_private.parquet",
        )
        statistic_frames = []
        for metric in metric_columns:
            subset = participant_anatomy[
                ["participant_id", "group_label", "match_set_id", metric]
            ].dropna()
            valid_sets = (
                subset.groupby("match_set_id")["group_label"].nunique()
            )
            subset = subset[
                subset["match_set_id"].isin(valid_sets[valid_sets == 2].index)
            ]
            if subset["match_set_id"].nunique() < 3:
                continue
            result = matched_set_permutation_inference(
                subset,
                [metric],
                permutations=2000,
                bootstrap_repetitions=1000,
            )
            result["analyzed_participants"] = int(subset["participant_id"].nunique())
            statistic_frames.append(result)
        if statistic_frames:
            roi_statistics = pd.concat(statistic_frames, ignore_index=True)

            def benjamini_hochberg(values):
                values = np.asarray(values, dtype=float)
                order = np.argsort(values)
                ranked = values[order]
                adjusted = np.minimum.accumulate(
                    (ranked * len(values) / np.arange(1, len(values) + 1))[::-1]
                )[::-1]
                output = np.empty_like(adjusted)
                output[order] = np.minimum(adjusted, 1.0)
                return output

            roi_statistics["fdr_q_value"] = benjamini_hochberg(
                roi_statistics["permutation_p_value"]
            )
            write_frame_atomic(
                roi_statistics,
                statistics_root / "anatomic_matched_set_inference.parquet",
            )
            display(roi_statistics.sort_values("fdr_q_value").round(5))

# Figure 1: cohort derivation and matching balance.
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].barh(derivation["stage"], derivation["participants"], color="#4C78A8")
axes[0].invert_yaxis()
axes[0].set_xlabel("Participants")
axes[0].set_title("Cohort derivation")
for index, value in enumerate(derivation["participants"]):
    axes[0].text(value, index, f" {int(value):,}", va="center")
before_age = participants[["age", "group_label"]].assign(stage="Before matching")
after_age = matched_participants[["age", "group_label"]].assign(stage="After matching")
for label, color in ((0, "#4C78A8"), (1, "#E45756")):
    axes[1].hist(
        after_age.loc[after_age["group_label"] == label, "age"],
        bins=20, alpha=0.45, density=True, color=color,
        label=widget("group_a_name") if label == 0 else widget("group_b_name"),
    )
axes[1].set_xlabel("Age at imaging (years)")
axes[1].set_ylabel("Density")
axes[1].set_title("Age distribution after matching")
axes[1].legend(frameon=False)
figure.tight_layout()
save_figure_atomic(figure, "figure_1_cohort_and_matching.png")
display(figure)
plt.close(figure)

# Figure 2: image quality by group.
figure, axis = plt.subplots(figsize=(6, 4.5))
labels = [widget("group_a_name"), widget("group_b_name")]
ordered_quality = quality_summary.set_index("group_label").reindex([0, 1])
axis.bar(labels, ordered_quality["quality_pass_rate"], color=["#4C78A8", "#E45756"])
axis.set_ylim(0, 1)
axis.set_ylabel("Quality-pass fraction")
axis.set_title("Technical image quality")
for index, value in enumerate(ordered_quality["quality_pass_rate"]):
    axis.text(index, value, f"{value:.1%}", ha="center", va="bottom")
figure.tight_layout()
save_figure_atomic(figure, "figure_2_quality_pass_rate.png")
display(figure)
plt.close(figure)

# Figure 3: held-out discrimination and calibration.
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve

labels_true = predictions["group_label"].to_numpy(int)
probabilities = predictions["group_b_probability_oof"].to_numpy(float)
fpr, tpr, _ = roc_curve(labels_true, probabilities)
precision, recall, _ = precision_recall_curve(labels_true, probabilities)
observed, predicted = calibration_curve(labels_true, probabilities, n_bins=10, strategy="quantile")
figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].plot(fpr, tpr, color="#4C78A8")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[0].set_title(f"Held-out ROC | AUROC={model_metrics['auroc']:.3f}")
axes[0].set_xlabel("False-positive rate")
axes[0].set_ylabel("True-positive rate")
axes[1].plot(recall, precision, color="#E45756")
axes[1].set_title(f"Precision–recall | AP={model_metrics['average_precision']:.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[2].plot(predicted, observed, marker="o", color="#72B7B2")
axes[2].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[2].set_title(f"Calibration | Brier={model_metrics['brier_score']:.3f}")
axes[2].set_xlabel("Mean predicted probability")
axes[2].set_ylabel("Observed group-B fraction")
figure.tight_layout()
save_figure_atomic(figure, "figure_3_classifier_performance.png")
display(figure)
plt.close(figure)

if len(roi_statistics):
    plot = roi_statistics.sort_values("group_b_minus_group_a")
    figure, axis = plt.subplots(figsize=(9, max(5, 0.35 * len(plot))))
    y = np.arange(len(plot))
    difference = plot["group_b_minus_group_a"].to_numpy(float)
    low = plot["bootstrap_95_ci_low"].to_numpy(float)
    high = plot["bootstrap_95_ci_high"].to_numpy(float)
    colors = np.where(plot["fdr_q_value"] < 0.05, "#C44E52", "#4C72B0")
    for index in range(len(plot)):
        axis.errorbar(
            difference[index], y[index],
            xerr=[[difference[index] - low[index]], [high[index] - difference[index]]],
            fmt="o", color=colors[index], capsize=3,
        )
    axis.axvline(0, color="black", linestyle="--", linewidth=1)
    axis.set_yticks(y)
    axis.set_yticklabels(plot["metric"].str.replace("_", " "), fontsize=8)
    axis.set_xlabel(f"{widget('group_b_name')} minus {widget('group_a_name')}")
    axis.set_title("Matched participant-level anatomic attribution")
    figure.tight_layout()
    save_figure_atomic(figure, "figure_4_anatomic_attribution.png")
    display(figure)
    plt.close(figure)

# Anatomically registered mean attribution maps, if maps and landmarks exist.
if len(explanation_status) and len(segmentation_status):
    registered = {0: [], 1: []}
    anatomy_valid = (
        explanation_status["anatomy_valid"].fillna(False)
        if "anatomy_valid" in explanation_status.columns
        else pd.Series(False, index=explanation_status.index)
    )
    valid_maps = explanation_status[
        explanation_status["explainability_ok"].fillna(False)
        & anatomy_valid
    ]
    import cv2
    for record in valid_maps.to_dict("records"):
        try:
            with np.load(record["attribution_map_path"]) as saved:
                grid = np.asarray(saved["variable_grid"], dtype=np.float32)
            heat = np.asarray(
                Image.fromarray(grid, mode="F").resize((256, 256), Image.Resampling.BILINEAR),
                dtype=np.float32,
            )
            matrix = disc_fovea_affine_matrix(
                [record["fovea_x_px"], record["fovea_y_px"],
                 record["optic_disc_x_px"], record["optic_disc_y_px"]],
                output_size=256,
            )
            registered[int(record["group_label"])].append(
                cv2.warpAffine(heat, matrix, (256, 256), flags=cv2.INTER_LINEAR)
            )
        except Exception:
            continue
    if all(registered[value] for value in (0, 1)):
        means = {value: np.mean(np.stack(registered[value]), axis=0) for value in (0, 1)}
        difference = means[1] - means[0]
        limit = max(float(np.quantile(np.abs(difference), 0.99)), 1e-8)
        figure, axes = plt.subplots(1, 3, figsize=(14, 4.5))
        for axis, value, title in (
            (axes[0], 0, widget("group_a_name")),
            (axes[1], 1, widget("group_b_name")),
        ):
            local_limit = max(float(np.quantile(np.abs(means[value]), 0.99)), 1e-8)
            axis.imshow(means[value], cmap="coolwarm", vmin=-local_limit, vmax=local_limit)
            axis.set_title(f"{title} mean | n={len(registered[value])} images")
        image = axes[2].imshow(difference, cmap="coolwarm", vmin=-limit, vmax=limit)
        axes[2].set_title(f"{widget('group_b_name')} minus {widget('group_a_name')}")
        figure.colorbar(image, ax=axes[2], fraction=0.046)
        for axis in axes:
            axis.axis("off")
        figure.suptitle("Disc–fovea registered exact RETFound attribution")
        figure.tight_layout()
        save_figure_atomic(figure, "figure_5_registered_mean_attribution.png")
        display(figure)
        plt.close(figure)

In [ ]:
# Stage 9: final PHI-free run report and durable completion marker.
group_counts = matched_participants.groupby("group_label")["participant_id"].nunique().to_dict()
quality_pass_counts = quality.groupby("group_label")["quality_pass"].sum().to_dict()
segmentation_count = (
    int(segmentation_status["segmentation_ok"].fillna(False).sum())
    if len(segmentation_status) else 0
)
explanation_count = (
    int(explanation_status["explainability_ok"].fillna(False).sum())
    if len(explanation_status) else 0
)
if not domain_rows:
    disease_warning = (
        "**Domain limitation:** device/source metadata were unavailable, so acquisition "
        "domain confounding could not be excluded."
    )
elif perfect_device_group_confounding:
    disease_warning = (
        "**Critical limitation:** group and device are perfectly confounded; the model "
        "cannot distinguish disease biology from acquisition domain."
    )
else:
    disease_warning = (
        "At least one device contained both groups. Review device-stratified counts and "
        "within-device sensitivity analyses before making disease-specific claims."
    )
report = f'''# Two-group retinal analysis: {analysis_name}

## Run

- Run ID: `{run_root.name}`
- Group A: **{widget('group_a_name')}** (`{widget('group_a_value')}`)
- Group B: **{widget('group_b_name')}** (`{widget('group_b_value')}`)
- Matching: 1:{int(widget('match_ratio'))}, age caliper ±{float(widget('age_caliper_years')):.2f} years
- Exact matching fields used: {', '.join(available_exact) if available_exact else 'none'}

## Completion summary

- Input images: {len(images):,}
- Quality-passing images: {int(quality['quality_pass'].fillna(False).sum()):,}
- Matched sets: {pairs['match_set_id'].nunique():,}
- Matched {widget('group_a_name')} participants: {int(group_counts.get(0, 0)):,}
- Matched {widget('group_b_name')} participants: {int(group_counts.get(1, 0)):,}
- Successful RETFound embeddings: {len(embeddings):,}
- Valid anatomy results: {segmentation_count:,}
- Successful exact explanations: {explanation_count:,}

## Held-out participant-level model

- AUROC: {model_metrics['auroc']:.3f} (95% CI {model_metrics['auroc_95_ci_low']:.3f}–{model_metrics['auroc_95_ci_high']:.3f})
- Average precision: {model_metrics['average_precision']:.3f} (95% CI {model_metrics['average_precision_95_ci_low']:.3f}–{model_metrics['average_precision_95_ci_high']:.3f})
- Brier score: {model_metrics['brier_score']:.3f}

## Interpretation safeguards

{disease_warning}

Matching, model evaluation, bootstrap confidence intervals, and anatomic
inference are participant-level. Matched sets remain intact across model
folds. Optic-disc and foveal outputs are localized circular ROIs derived
from predicted landmarks; only the vessel output is a pixel segmentation.
Explainability localizes evidence used by the fitted linear head but does
not independently prove causal physiology.

## Output guide

- `00_config`: nonsecret resolved configuration
- `01_input_manifest`: normalized private image manifest and input checkpoints
- `02_quality`: quality batches and aggregate quality summary
- `03_age_matching`: private pairs/membership and aggregate balance
- `04_retfound_embeddings`: resumable embedding status and vectors
- `05_segmentations`: masks, overlays, and batch manifests
- `06_participant_model`: grouped OOF predictions, model heads, and metrics
- `07_explainability`: exact maps, overlays, and optional occlusion results
- `08_statistics`: aggregate and participant-level analysis tables
- `09_figures`: PHI-free publication-oriented figures

Credentials were not persisted. Files ending in `_private` contain paths or
participant identifiers and should remain within the governed environment.
'''
report = "\n".join(line[8:] if line.startswith("        ") else line for line in report.splitlines())
local_report = local_staging / f"{uuid.uuid4().hex}_RUN_README.md"
local_report.write_text(report, encoding="utf-8")
publish_local_artifact(local_report, run_root / "RUN_README.md")
local_report.unlink(missing_ok=True)
write_json_atomic(
    {
        "status": "complete",
        "run_id": run_root.name,
        "completed_unix_seconds": time.time(),
        "run_readme": str(run_root / "RUN_README.md"),
    },
    run_root / "_SUCCESS.json",
)
print("Pipeline complete")
record_progress("pipeline", "complete", 1, 1)
print("Run README:", run_root / "RUN_README.md")
print("Figures:", figure_root)